In [3]:
import pandas as pd
data = pd.read_csv("TMDB_movie_dataset_v11.csv")
data.info()

NameError: name 'pd' is not defined

In [3]:
def report(df):
    return pd.DataFrame({
            "dtype": df.dtypes,
            "missing_count": df.isna().sum(),
            "missing_pct": df.isna().mean() * 100,
            "unique_values": df.nunique(),
        }).sort_values(by="missing_pct", ascending=False)

In [4]:
before_report = report(data)
before_report

,dtype,missing_count,missing_pct,unique_values
homepage,object,1232497,89.676973,132633
tagline,object,1182944,86.071477,183114
keywords,object,1030350,74.968677,198606
backdrop_path,object,1029368,74.897226,341475
production_companies,object,784116,57.052593,231765
imdb_id,object,716370,52.123367,654614
production_countries,object,657182,47.816824,10933
spoken_languages,object,631398,45.940770,7442
genres,object,596583,43.407617,14784
poster_path,object,479831,34.912695,888917


In [1]:
# Clean Data
movies = data.copy()

# replace empty data with NaN
movies.replace(r'^\s*$', pd.NA, regex=True, inplace=True)

# replace invalid values with NaN
movies.loc[movies['runtime'] <= 0, 'runtime'] = pd.NA
movies.loc[movies['budget'] < 0, 'budget'] = pd.NA
movies.loc[movies['revenue'] < 0, 'revenue'] = pd.NA

# convert date strings to datetime
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

# ensure numeric values
int_cols = ['id', 'budget', 'revenue', 'runtime', 'vote_count']
float_cols = ['vote_average', 'popularity']
movies[int_cols] = movies[int_cols].apply(pd.to_numeric, errors='coerce').astype('Int64')
movies[float_cols] = movies[float_cols].apply(pd.to_numeric, errors='coerce')

# fill missing values, if possible
movies['title'] = movies['title'].fillna(movies['original_title'])
movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())

# drop unusable data
movies = movies[movies['title'].notna()]
movies = movies[movies['id'].notna()]

# remove duplicates
movies.drop_duplicates(subset=['id'], inplace=True)

# only look at released movies
movies = movies[movies['status'] == 'Released']

# drop irrelevant columns
cols_to_drop = ['homepage', 'tagline', 'keywords', 'backdrop_path', 'imdb_id', 'poster_path', 'overview']
analysis_movies = movies.drop(columns=cols_to_drop)

NameError: name 'data' is not defined

In [10]:
after_report = report(analysis_movies)
print(f"Total rows: {analysis_movies.shape[0]}")
after_report

Total rows: 1326652


,dtype,missing_count,missing_pct,unique_values
production_companies,object,761507,57.400660,218133
production_countries,object,638565,48.133572,10336
spoken_languages,object,616526,46.472323,7262
genres,object,582820,43.931641,14219
release_date,datetime64[ns],265266,19.995146,43164
revenue,Int64,1,0.000075,14558
original_language,object,0,0.000000,177
popularity,float64,0,0.000000,20140
original_title,object,0,0.000000,1169793
id,Int64,0,0.000000,1326652


In [11]:
before_missing_pct = before_report.drop(index=cols_to_drop)[['missing_pct']].round(2)
after_missing_pct = after_report[['missing_pct']].round(2)

comparison = before_missing_pct.join(
    after_missing_pct,
    lsuffix="_before",
    rsuffix="_after"
)

comparison

,missing_pct_before,missing_pct_after
production_companies,57.05,57.40
production_countries,47.82,48.13
spoken_languages,45.94,46.47
genres,43.41,43.93
release_date,21.34,20.00
title,0.00,0.00
original_title,0.00,0.00
vote_average,0.00,0.00
vote_count,0.00,0.00
revenue,0.00,0.00


In [12]:
# export
analysis_movies.to_csv("cleaned_movies.csv", index=False)